<img src="https://upload.wikimedia.org/wikipedia/commons/b/b0/Logo_Universidad_Polit%C3%A9cnica_Salesiana_del_Ecuador.png" alt="logo universidad" width="550" height="150">

### Materia Vision por Computador

### Practica 4 — Notebook 03: Benchmark de hardware CPU vs GPU (Google Colab)

---

**Integrantes:** Paúl Sebastián Naspud, Jennyfer Camila Ramírez

**Fecha de Entrega:**

---

Este cuaderno mide **exclusivamente rendimiento de hardware** (tiempos de inferencia, FPS, uso de
CPU/RAM/GPU/VRAM) al correr el modelo ya entrenado (`best.pt`, YOLO26-seg, dataset *Tools
Segmentation*) en Google Colab. No se reentrena el modelo ni se calculan métricas de calidad
(mAP, precision, recall) — eso ya se evaluó en el Notebook 01.


**Contenido:**


0. Ingesta de datos (Drive, pesos, dataset).
1. Utilidades de timing (cold start / warm-up / steady-state) y sampler de recursos.
2. Barrido sistemático device × batch × resolución × precisión.
3. Escenario de carga sostenida.
4. Métricas derivadas y gráficas comparativas.
5. Video: inferencia sobre un clip subido + ráfaga de fotos por webcam.
6. Ficha técnica y conclusiones.


## 0. Ingesta de datos: pesos del modelo y dataset

El archivo de pesos entrenados (`best.pt`) se carga en la sesion mediante el selector de archivos
de Colab

In [ ]:
!pip install -q -U roboflow ultralytics psutil


In [ ]:
from google.colab import files

print("Selector de archivo — pesos entrenados (best.pt):")
subido_pesos = files.upload()
MODEL_WEIGHTS = next(iter(subido_pesos.keys()))
print("Pesos recibidos:", MODEL_WEIGHTS)


In [ ]:
from pathlib import Path

RESULTS_DIR = Path("resultados_benchmark")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Identifica la sesion actual de Colab (la GPU asignada no es la misma entre sesiones).
# Cambiar antes de cada corrida nueva si se quiere comparar varias sesiones (punto 1 del plan).
SESSION_LABEL = "sesion_1"

print("Resultados de esta sesion:", RESULTS_DIR.resolve(), "| etiqueta:", SESSION_LABEL)


In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="t87DRxOynZJMDJVpZltW")
project = rf.workspace("paul-space-qcfcl").project("tools-segmentation-f2nhg-tf2cd")
dataset = project.version(1).download("coco-segmentation")

DATASET_DIR = Path(dataset.location)
BENCH_SPLIT = "test" if (DATASET_DIR / "test").exists() else "valid"
print("Dataset descargado en:", DATASET_DIR)
print("Split usado para el benchmark:", BENCH_SPLIT)


In [ ]:
import glob

bench_images_all = sorted(
    glob.glob(str(DATASET_DIR / BENCH_SPLIT / "*.jpg"))
    + glob.glob(str(DATASET_DIR / BENCH_SPLIT / "*.JPEG"))
    + glob.glob(str(DATASET_DIR / BENCH_SPLIT / "*.jpeg"))
)
print(f"Imagenes disponibles en '{BENCH_SPLIT}':", len(bench_images_all))

# Subconjunto representativo para el barrido (punto 3 del plan): no hace falta el set completo
# para caracterizar rendimiento, y mantiene manejable el tiempo total de la sesion.
N_BENCH_IMAGES = min(100, len(bench_images_all))
bench_images = bench_images_all[:N_BENCH_IMAGES]
print("Imagenes usadas en el barrido:", len(bench_images))


## 1. Utilidades de timing y monitoreo de recursos

Reglas de medicion :

- Reloj monotonico `time.perf_counter()`, nunca `time.time()`.
- En GPU, `torch.cuda.synchronize()` antes y despues de cada region cronometrada — las llamadas
  CUDA son asincronas y sin sincronizar el tiempo medido queda subestimado.
- Tres fases separadas: **cold start** (carga del modelo + primera prediccion, se reporta aparte),
  **warm-up** (iteraciones descartadas) y **steady-state** (lo que se reporta: media, mediana,
  desviacion estandar y percentiles p90/p99, no solo el promedio).


In [ ]:
import threading
import time
import subprocess

import numpy as np
import pandas as pd
import psutil
import torch


def gpu_snapshot():
    """Uso puntual de GPU via nvidia-smi (utilizacion % y memoria usada/total en MB)."""
    if not torch.cuda.is_available():
        return None, None, None
    try:
        salida = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,noheader,nounits"],
            text=True,
        ).strip()
        util, mem_used, mem_total = [float(x) for x in salida.split(",")]
        return util, mem_used, mem_total
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None, None, None


class ResourceSampler:
    """Muestrea CPU%, RAM y GPU%/VRAM en un hilo de fondo, sin bloquear el bucle medido."""

    def __init__(self, interval_s=0.2):
        self.interval_s = interval_s
        self._stop_event = threading.Event()
        self._thread = None
        self.rows = []

    def _run(self):
        while not self._stop_event.is_set():
            t = time.perf_counter()
            cpu_pct = psutil.cpu_percent(interval=None)
            ram = psutil.virtual_memory()
            gpu_util, gpu_mem_used, gpu_mem_total = gpu_snapshot()
            self.rows.append({
                "t": t,
                "cpu_pct": cpu_pct,
                "ram_pct": ram.percent,
                "gpu_util_pct": gpu_util,
                "gpu_mem_used_mb": gpu_mem_used,
                "gpu_mem_total_mb": gpu_mem_total,
            })
            time.sleep(self.interval_s)

    def start(self):
        self.rows = []
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=2 * self.interval_s)
        return pd.DataFrame(self.rows)


print("Utilidades de timing y ResourceSampler listas.")


In [ ]:
def predict_timed(model, images, device, imgsz, half=False):
    """Corre una prediccion sobre `images` y devuelve el tiempo en ms, sincronizando CUDA si aplica."""
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    model.predict(images, device=device, imgsz=imgsz, half=half, verbose=False, conf=0.4)
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000.0


def medir_cold_start(weights_path, device, imgsz=640):
    """Carga del modelo + primera prediccion, aislado del resto de las mediciones."""
    t0 = time.perf_counter()
    m = YOLO(weights_path)
    t_carga_ms = (time.perf_counter() - t0) * 1000.0

    t_primera_ms = predict_timed(m, [bench_images[0]], device, imgsz)
    m.fuse()  # optimizacion de una sola vez, fuera de cualquier bucle cronometrado
    return m, t_carga_ms, t_primera_ms


## 2. Barrido sistematico: device × batch × resolucion × precision

Matriz de factores (una fila de resultados por combinacion, con `N_REPETICIONES` iteraciones
"steady-state" cada una, tras su propio mini warm-up):

| Factor | Valores | Nota |
|---|---|---|
| device | `cuda` (si esta disponible), `cpu` | comparacion central |
| batch | 1, 4, 8, 16, 32 | CPU tiende a escalar plano/peor; GPU casi lineal hasta saturar |
| imgsz | 320, 640, 960, 1280 | costo ~cuadratico con la resolucion |
| precision | `fp32` (ambos devices), `fp16` (solo GPU) | optimizacion extra, barata, exclusiva de GPU |

Cada combinacion de forma de entrada (batch, imgsz) tiene su propio costo de "primera vez"
(autotune de cuDNN por shape), asi que se descartan un par de iteraciones por combinacion ademas
del warm-up global del device.

Los resultados se guardan **incrementalmente** a CSV apenas se calcula cada fila, para no perder
todo si Colab se desconecta a mitad del barrido.


In [ ]:
from ultralytics import YOLO

DEVICES_TO_TEST = ["cuda", "cpu"] if torch.cuda.is_available() else ["cpu"]
BATCH_SIZES = [1, 4, 8, 16, 32]
IMG_SIZES = [320, 640, 960, 1280]
N_WARMUP_ITERS = 5       # descartadas, al cargar el modelo en cada device
N_WARMUP_PER_CONFIG = 2  # descartadas, por cada combinacion (batch, imgsz) nueva
N_REPETICIONES = 8       # iteraciones "steady-state" registradas por combinacion

print("Devices a evaluar en esta sesion:", DEVICES_TO_TEST)
print("NOTA (rigor metodologico): para aislar por completo CPU de GPU, lo mas correcto es correr")
print("este notebook una vez con DEVICES_TO_TEST=['cuda'] y, tras reiniciar el entorno de")
print("ejecucion (Entorno de ejecucion > Reiniciar sesion), correrlo de nuevo con ['cpu'].")
print("Por defecto aqui se corren ambos en la misma sesion para simplicidad de un solo run.")


In [ ]:
import itertools

resultados_sweep = []
cold_start_rows = []
csv_path = RESULTS_DIR / f"sweep_{SESSION_LABEL}.csv"

for device in DEVICES_TO_TEST:
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    model, t_carga_ms, t_primera_ms = medir_cold_start(MODEL_WEIGHTS, device)
    cold_start_rows.append({
        "session": SESSION_LABEL, "device": device,
        "carga_modelo_ms": t_carga_ms, "primera_prediccion_ms": t_primera_ms,
    })
    print(f"[{device}] cold start -> carga: {t_carga_ms:.1f} ms | primera prediccion: {t_primera_ms:.1f} ms")

    for _ in range(N_WARMUP_ITERS):
        predict_timed(model, bench_images[:1], device, 640)

    precisiones = ["fp32", "fp16"] if device == "cuda" else ["fp32"]

    for imgsz, batch, precision in itertools.product(IMG_SIZES, BATCH_SIZES, precisiones):
        half = precision == "fp16"
        imagenes_lote = list(itertools.islice(itertools.cycle(bench_images), batch))

        for _ in range(N_WARMUP_PER_CONFIG):
            predict_timed(model, imagenes_lote, device, imgsz, half=half)

        if device == "cuda":
            torch.cuda.reset_peak_memory_stats()

        for rep in range(N_REPETICIONES):
            lat_ms = predict_timed(model, imagenes_lote, device, imgsz, half=half)
            resultados_sweep.append({
                "session": SESSION_LABEL, "device": device, "batch": batch, "imgsz": imgsz,
                "precision": precision, "rep": rep,
                "latencia_ms": lat_ms, "latencia_ms_por_imagen": lat_ms / batch,
            })

        peak_vram_mb = torch.cuda.max_memory_allocated() / 1e6 if device == "cuda" else None
        if resultados_sweep:
            resultados_sweep[-1]["peak_vram_mb"] = peak_vram_mb

        # guardado incremental
        pd.DataFrame(resultados_sweep).to_csv(csv_path, index=False)

    del model
    if device == "cuda":
        torch.cuda.empty_cache()

df_sweep = pd.DataFrame(resultados_sweep)
df_cold_start = pd.DataFrame(cold_start_rows)
print("Barrido terminado. Filas:", len(df_sweep))
df_sweep.head()


## 3. Escenario de carga sostenida (stress test)

Se corre inferencia en bucle continuo durante varios minutos sobre una configuracion fija y
representativa (batch=1, resolucion nativa), muestreando recursos con `ResourceSampler` en
paralelo. El objetivo es ver si la latencia se degrada con el tiempo (throttling, contencion en
una GPU compartida de la nube) — algo que un solo promedio puntual no muestra.


In [ ]:
STRESS_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STRESS_DURATION_S = 120  # ajustar a 2-3 minutos segun tiempo disponible de sesion
STRESS_IMGSZ = 640

stress_model = YOLO(MODEL_WEIGHTS)
stress_model.fuse()
for _ in range(N_WARMUP_ITERS):
    predict_timed(stress_model, bench_images[:1], STRESS_DEVICE, STRESS_IMGSZ)

sampler = ResourceSampler(interval_s=0.5)
sampler.start()

stress_rows = []
t_inicio = time.perf_counter()
i = 0
while (time.perf_counter() - t_inicio) < STRESS_DURATION_S:
    img = bench_images[i % len(bench_images)]
    t_rel = time.perf_counter() - t_inicio
    lat_ms = predict_timed(stress_model, [img], STRESS_DEVICE, STRESS_IMGSZ)
    stress_rows.append({"t_relativo_s": t_rel, "latencia_ms": lat_ms})
    i += 1

df_recursos_stress = sampler.stop()
df_stress = pd.DataFrame(stress_rows)
df_stress.to_csv(RESULTS_DIR / f"stress_{SESSION_LABEL}.csv", index=False)
df_recursos_stress.to_csv(RESULTS_DIR / f"stress_recursos_{SESSION_LABEL}.csv", index=False)

del stress_model
if STRESS_DEVICE == "cuda":
    torch.cuda.empty_cache()

print(f"Carga sostenida ({STRESS_DEVICE}): {len(df_stress)} inferencias en {STRESS_DURATION_S} s")


In [ ]:
import matplotlib.pyplot as plt

COLOR_GPU = "#2a78d6"
COLOR_CPU = "#1baf7a"
color_stress = COLOR_GPU if STRESS_DEVICE == "cuda" else COLOR_CPU

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_stress["t_relativo_s"], df_stress["latencia_ms"], color=color_stress, linewidth=1)
ax.set_xlabel("Tiempo transcurrido (s)")
ax.set_ylabel("Latencia por inferencia (ms)")
ax.set_title(f"Carga sostenida — {STRESS_DEVICE.upper()} ({STRESS_DURATION_S}s continuos)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"stress_latencia_{SESSION_LABEL}.png", dpi=150)
plt.show()


## 4. Metricas derivadas y graficas comparativas

Todo sale de `df_sweep` (formato largo: session, device, batch, imgsz, precision, rep,
latencia_ms, ...) — una sola fuente de verdad para las estadisticas y todas las graficas.


In [ ]:
resumen = (
    df_sweep.groupby(["device", "batch", "imgsz", "precision"])["latencia_ms_por_imagen"]
    .agg(
        media_ms="mean",
        mediana_ms="median",
        std_ms="std",
        p90_ms=lambda s: s.quantile(0.90),
        p99_ms=lambda s: s.quantile(0.99),
    )
    .reset_index()
)
resumen["fps"] = 1000.0 / resumen["media_ms"]
resumen["fps_cv"] = resumen["std_ms"] / resumen["media_ms"]  # coeficiente de variacion (estabilidad)

resumen.to_csv(RESULTS_DIR / f"resumen_{SESSION_LABEL}.csv", index=False)
resumen.sort_values(["imgsz", "batch", "device"]).head(20)


In [ ]:
# Speedup GPU vs CPU (fp32) por cada combinacion de batch/imgsz
base = resumen[resumen["precision"] == "fp32"].pivot_table(
    index=["batch", "imgsz"], columns="device", values="media_ms"
)
if "cuda" in base.columns and "cpu" in base.columns:
    base["speedup_gpu_vs_cpu"] = base["cpu"] / base["cuda"]
    print(base[["cpu", "cuda", "speedup_gpu_vs_cpu"]])
else:
    print("Se necesitan filas de 'cpu' y 'cuda' en la misma sesion para calcular speedup directo.")
    print("Si se corrieron en sesiones separadas, comparar 'resumen_<sesion>.csv' de cada una.")


In [ ]:
# Throughput (imagenes/seg) vs batch size, a resolucion fija — CPU vs GPU superpuestos
IMGSZ_REF = 640
fig, ax = plt.subplots(figsize=(6, 4))

for device, color in (("cuda", COLOR_GPU), ("cpu", COLOR_CPU)):
    sub = resumen[(resumen["device"] == device) & (resumen["imgsz"] == IMGSZ_REF) & (resumen["precision"] == "fp32")]
    if not sub.empty:
        sub = sub.sort_values("batch")
        ax.plot(sub["batch"], sub["fps"], marker="o", color=color, label=device.upper())

ax.set_xlabel("Batch size")
ax.set_ylabel("Throughput (imagenes/seg)")
ax.set_title(f"Throughput vs batch size (imgsz={IMGSZ_REF})")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"throughput_vs_batch_{SESSION_LABEL}.png", dpi=150)
plt.show()


In [ ]:
# Latencia vs resolucion, a batch fijo — CPU vs GPU superpuestos
BATCH_REF = 1
fig, ax = plt.subplots(figsize=(6, 4))

for device, color in (("cuda", COLOR_GPU), ("cpu", COLOR_CPU)):
    sub = resumen[(resumen["device"] == device) & (resumen["batch"] == BATCH_REF) & (resumen["precision"] == "fp32")]
    if not sub.empty:
        sub = sub.sort_values("imgsz")
        ax.plot(sub["imgsz"], sub["media_ms"], marker="o", color=color, label=device.upper())

ax.set_xlabel("Resolucion de entrada (imgsz)")
ax.set_ylabel("Latencia media (ms/imagen)")
ax.set_title(f"Latencia vs resolucion (batch={BATCH_REF})")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"latencia_vs_resolucion_{SESSION_LABEL}.png", dpi=150)
plt.show()


In [ ]:
# FP32 vs FP16 en GPU (si hay GPU disponible)
gpu_precision = resumen[(resumen["device"] == "cuda") & (resumen["imgsz"] == IMGSZ_REF)]
if not gpu_precision.empty:
    fig, ax = plt.subplots(figsize=(6, 4))
    for precision, color in (("fp32", COLOR_GPU), ("fp16", "#1baf7a")):
        sub = gpu_precision[gpu_precision["precision"] == precision].sort_values("batch")
        if not sub.empty:
            ax.plot(sub["batch"], sub["fps"], marker="o", color=color, label=precision.upper())
    ax.set_xlabel("Batch size")
    ax.set_ylabel("Throughput (imagenes/seg)")
    ax.set_title(f"GPU: FP32 vs FP16 (imgsz={IMGSZ_REF})")
    ax.legend(frameon=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"fp32_vs_fp16_{SESSION_LABEL}.png", dpi=150)
    plt.show()
else:
    print("No hay GPU disponible en esta sesion; se omite la comparacion FP32 vs FP16.")


In [ ]:
# VRAM pico por batch/imgsz (define el batch maximo viable en la GPU asignada)
vram_rows = df_sweep[df_sweep["device"] == "cuda"].dropna(subset=["peak_vram_mb"])
if not vram_rows.empty:
    vram_summary = vram_rows.groupby(["batch", "imgsz"])["peak_vram_mb"].max().reset_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    for imgsz in sorted(vram_summary["imgsz"].unique()):
        sub = vram_summary[vram_summary["imgsz"] == imgsz].sort_values("batch")
        ax.plot(sub["batch"], sub["peak_vram_mb"], marker="o", label=f"imgsz={imgsz}")
    ax.set_xlabel("Batch size")
    ax.set_ylabel("VRAM pico (MB)")
    ax.set_title("VRAM pico por configuracion")
    ax.legend(frameon=False, fontsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"vram_pico_{SESSION_LABEL}.png", dpi=150)
    plt.show()
else:
    print("No hay datos de VRAM (no se uso GPU en esta sesion).")


## 5. Video: inferencia sobre un clip subido + rafaga de fotos por webcam

Colab no soporta `cv2.imshow` ni streaming nativo de webcam, asi que la parte de video se resuelve
de dos formas complementarias:

- **Principal:** se sube un clip corto y se procesa cuadro a cuadro, guardando el video anotado
  con overlay de FPS — da una curva de FPS real y sostenida sobre muchos cuadros (tambien sirve
  como una segunda muestra de carga sostenida, sobre datos reales de video en vez de imagenes
  fijas repetidas).
- **Complementaria:** una rafaga de fotos via el snippet JS `take_photo()` (el mismo del Notebook
  01), para medir la latencia *interactiva* real -incluyendo el viaje navegador <-> Colab-,
  distinta de la latencia de inferencia pura medida en las secciones anteriores.


In [ ]:
from google.colab import files

print("Selector de archivo — clip de video corto (mp4/avi):")
subido = files.upload()
video_path = next(iter(subido.keys()))
print("Video recibido:", video_path)


In [ ]:
import cv2

VIDEO_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

video_model = YOLO(MODEL_WEIGHTS)
video_model.fuse()

cap = cv2.VideoCapture(video_path)
fps_video = cap.get(cv2.CAP_PROP_FPS) or 25.0
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out_path = str(RESULTS_DIR / f"video_anotado_{SESSION_LABEL}.mp4")
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps_video, (w, h))

frame_latencias = []
while True:
    ret, frame = cap.read()
    if not ret:
        break

    if VIDEO_DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    result = video_model.predict(frame, device=VIDEO_DEVICE, imgsz=640, verbose=False, conf=0.4)[0]
    if VIDEO_DEVICE == "cuda":
        torch.cuda.synchronize()
    lat_ms = (time.perf_counter() - t0) * 1000.0

    annotated = result.plot()
    cv2.putText(annotated, f"{VIDEO_DEVICE.upper()} - {1000.0 / lat_ms:.1f} FPS", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    writer.write(annotated)
    frame_latencias.append(lat_ms)

cap.release()
writer.release()
del video_model
if VIDEO_DEVICE == "cuda":
    torch.cuda.empty_cache()

df_video = pd.DataFrame({"frame": range(len(frame_latencias)), "latencia_ms": frame_latencias})
df_video.to_csv(RESULTS_DIR / f"video_latencias_{SESSION_LABEL}.csv", index=False)

print(f"Video anotado guardado en: {out_path}")
print(f"Cuadros procesados: {len(frame_latencias)} | FPS medio: {1000.0 / df_video['latencia_ms'].mean():.1f}")


In [ ]:
files.download(out_path)


In [ ]:
from base64 import b64decode

from google.colab.output import eval_js
from IPython.display import Javascript
from IPython.display import display as ipy_display


def take_photo(filename="webcam.jpg", quality=0.9):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capturar';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    ipy_display(js)
    data = eval_js(f"takePhoto({quality})")
    binary = b64decode(data.split(",")[1])
    with open(filename, "wb") as f:
        f.write(binary)
    return filename


N_FOTOS_RAFAGA = 5
rafaga_rows = []
foto_model = YOLO(MODEL_WEIGHTS)
foto_model.fuse()

for i in range(N_FOTOS_RAFAGA):
    t0 = time.perf_counter()
    foto_path = take_photo(filename=f"webcam_{i}.jpg")
    t_captura_ms = (time.perf_counter() - t0) * 1000.0

    lat_ms = predict_timed(foto_model, [foto_path], VIDEO_DEVICE, 640)
    rafaga_rows.append({
        "foto": i, "captura_interactiva_ms": t_captura_ms, "inferencia_pura_ms": lat_ms,
    })

del foto_model
if VIDEO_DEVICE == "cuda":
    torch.cuda.empty_cache()

df_rafaga = pd.DataFrame(rafaga_rows)
df_rafaga.to_csv(RESULTS_DIR / f"rafaga_webcam_{SESSION_LABEL}.csv", index=False)
print("Latencia interactiva (captura navegador incluida) vs inferencia pura:")
df_rafaga


## 6. Ficha tecnica y conclusiones


In [ ]:
import sys

import ultralytics

ficha = {
    "session_label": SESSION_LABEL,
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_torch": torch.version.cuda,
    "ultralytics": ultralytics.__version__,
    "gpu_disponible": torch.cuda.is_available(),
    "num_threads_cpu_torch": torch.get_num_threads(),
    "cpu_nucleos_logicos": psutil.cpu_count(logical=True),
    "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 2),
}

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    ficha.update({
        "gpu_nombre": props.name,
        "gpu_vram_total_gb": round(props.total_memory / 1e9, 2),
        "gpu_compute_capability": f"{props.major}.{props.minor}",
    })

for k, v in ficha.items():
    print(f"{k:24s}: {v}")

pd.DataFrame([ficha]).to_csv(RESULTS_DIR / f"ficha_tecnica_{SESSION_LABEL}.csv", index=False)


**Conclusiones (completar tras correr el cuaderno, idealmente en 2+ sesiones de Colab):**

- **GPU asignada por sesion y su impacto:** ...
- **Speedup GPU vs CPU** (fp32, por resolucion/batch): ...
- **Punto de saturacion de throughput** al aumentar el batch, en cada device: ...
- **Costo de la resolucion** (`imgsz`) en cada device: ...
- **Beneficio de FP16 en GPU:** ...
- **Estabilidad de FPS** en carga sostenida (¿hay degradacion/throttling visible?): ...
- **VRAM pico** observada y batch maximo viable en la GPU asignada: ...
- **Latencia interactiva (webcam) vs inferencia pura:** ...


In [ ]:
# Cargar y combinar los CSV de 'resumen_<sesion>.csv' de todas las sesiones corridas hasta ahora,
# para reportar un rango (no un solo numero puntual) tal como pide el punto 1 del plan.
resumen_paths = sorted(RESULTS_DIR.glob("resumen_*.csv"))
if len(resumen_paths) > 1:
    df_todas_sesiones = pd.concat([pd.read_csv(p) for p in resumen_paths], ignore_index=True)
    rango_por_config = (
        df_todas_sesiones.groupby(["device", "batch", "imgsz", "precision"])["media_ms"]
        .agg(["min", "max", "mean"])
        .reset_index()
    )
    print(f"Comparando {len(resumen_paths)} sesiones: {[p.name for p in resumen_paths]}")
    rango_por_config.head(20)
else:
    print("Todavia hay una sola sesion guardada. Correr el notebook de nuevo (idealmente tras")
    print("reiniciar el entorno de ejecucion) con otro SESSION_LABEL para comparar el rango")
    print("entre sesiones, ya que Colab no garantiza la misma GPU cada vez.")


### Descarga de resultados de la sesion

El almacenamiento de la VM de Colab es efimero: esta celda comprime `resultados_benchmark/`
(CSV, graficas, video anotado, ficha tecnica) y descarga el archivo comprimido al equipo local.


In [ ]:
import shutil

zip_base = f"resultados_practica4_{SESSION_LABEL}"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=RESULTS_DIR)
files.download(zip_path)
print("Resultados comprimidos y descargados desde:", zip_path)
